# Boulder recommendation  
## Similarity calculation taking into account ascents and boulder features (grade, styles)  
Ascents: Jaccard similarity  
Grades: Cosine similarity  
Styles: Dot product normalized with the highest style sharing count 

# Status: Recommendation system satifying, can be put in production

## SQLAlchemy session creation

In [1]:
import numpy as np
import pandas as pd

from sqlalchemy.orm import Session
from sqlalchemy import create_engine

DB_URL = "sqlite:///../../matchit.db"

engine = create_engine(DB_URL, echo=False)

session = Session(engine)

In [2]:
import sys

sys.path.append("../../")

## Similarity matrix training **based on similar repetitors**

### Database Query

In [ ]:
from sqlalchemy import select
from models.ascent import Ascent
from models.boulder import Boulder

ascents = session.execute(
    select(Ascent.user_id, Ascent.boulder_id)
).all()
ascents_df = pd.DataFrame(data=ascents, columns=["user_id", "id"])


[<Boulder(name: Galileo bis, grade: 6A, Ascents: []),
 <Boulder(name: Un uomo un perchè, grade: 6A, Ascents: []),
 <Boulder(name: N.N., grade: 6A, Ascents: []),
 <Boulder(name: Die Roboter, grade: 6A, Ascents: []),
 <Boulder(name: Fast Food, grade: 6A, Ascents: [])]

,user_id,id


### boulder_user matrix (Pivot table)

In [4]:
boulder_user_matrix = ascents_df.pivot_table(
    index="id",
    columns="user_id",
    aggfunc="size",
    fill_value=0,
    dropna=True,
)
# boulder_user_matrix = boulder_user_matrix[boulder_user_matrix.index < 20]
boulder_ids = boulder_user_matrix.index

In [5]:
display(boulder_user_matrix)

user_id
id


### Conversion to sparse matrix

In [9]:
from scipy.sparse import coo_matrix

boulder_user_matrix = coo_matrix(boulder_user_matrix)
print(boulder_user_matrix)

<COOrdinate sparse matrix of dtype 'int64'
	with 559233 stored elements and shape (28497, 7263)>
  Coords	Values
  (0, 1)	1
  (0, 2)	1
  (0, 3)	1
  (0, 4)	1
  (0, 5)	1
  (0, 6)	1
  (1, 8)	1
  (1, 9)	1
  (1, 10)	1
  (1, 11)	1
  (2, 13)	1
  (2, 14)	1
  (2, 15)	1
  (3, 17)	1
  (4, 7)	1
  (4, 19)	1
  (4, 20)	1
  (5, 17)	1
  (6, 1)	1
  (6, 3)	1
  (6, 4)	1
  (6, 5)	1
  (6, 13)	1
  (6, 17)	1
  (6, 21)	1
  :	:
  (28490, 489)	1
  (28490, 953)	1
  (28491, 364)	1
  (28491, 574)	1
  (28491, 1202)	1
  (28491, 1318)	1
  (28491, 1385)	1
  (28491, 3589)	1
  (28492, 364)	1
  (28492, 574)	1
  (28492, 1202)	1
  (28492, 1318)	1
  (28492, 3589)	1
  (28493, 364)	1
  (28493, 489)	1
  (28494, 364)	1
  (28494, 1318)	1
  (28494, 1777)	1
  (28494, 3589)	1
  (28495, 364)	1
  (28495, 1318)	1
  (28495, 1777)	1
  (28495, 3589)	1
  (28496, 364)	1
  (28496, 2431)	1


### Ascents similarity training

In [10]:
def jaccard_pairwise_similarity(X):
    # CSR matrix storing the number of shared ascents for each pair of
    # boulders sharing at least one ascent

    intersection = X @ X.T

    # 1D array storing the total number of ascent for each boulder
    row_sums = np.asarray(X.sum(axis=1)).ravel()

    # intersection decomposition for calculation on 1D arrays
    rows, cols = intersection.nonzero()
    intersection_data = intersection.data

    union = row_sums[rows] + row_sums[cols] - intersection_data

    jaccard = intersection_data / union

    # Index remapping based on the boulder ids
    new_rows = boulder_ids[rows]
    new_cols = boulder_ids[cols]

    return coo_matrix(
        (jaccard, (new_rows, new_cols)),
        dtype=np.float32,
    )

similarity_ascents = jaccard_pairwise_similarity(boulder_user_matrix)

sparsity = 1.0 - similarity_ascents.nnz / (
    similarity_ascents.shape[0] * similarity_ascents.shape[1]
)

print(f"Sparsity: {sparsity:.2%}")

Sparsity: 90.44%


In [11]:
# similarity_ascents_df = pd.DataFrame(similarity_ascents.toarray())
# display(similarity_ascents_df)

## Similarity matrix training **based on similar features**

### Database query and dataframe creation

In [12]:
from sqlalchemy.orm import joinedload
from models.boulder import Boulder

# Database request
boulders = (
    session.scalars(
        select(Boulder).options(
            joinedload(Boulder.grade), joinedload(Boulder.styles)
        )
    )
    .unique()
    .all()
)

# Data extraction
boulders = [
    {
        "id": boulder.id,
        "grade": boulder.grade.correspondence,
        "styles": [style.style for style in boulder.styles],
    }
    for boulder in boulders
]

# Dataframe setup
boulders_df = pd.DataFrame(boulders)
# boulders_df = boulders_df[boulders_df.id < 20]
boulder_ids = boulders_df.id
boulders_df.head()


,id,grade,styles
0,1,32,"[slightly overhanging, sitstart]"
1,2,28,[slopers]
2,3,29,"[slopers, wall]"
3,4,30,"[slightly overhanging, sitstart, traverse]"
4,5,30,"[slopers, wall, crimps]"


### Style

#### Binarizing

In [13]:
from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer()
styles = mlb.fit_transform(boulders_df.styles)

# styles_df = pd.DataFrame(styles, columns=mlb.classes_)
# display(styles_df)

#### Conversion to sparse matrix

In [14]:
styles = coo_matrix(styles, dtype=np.float32)

#### Style similarity training

In [15]:
# Dot product
similarity_style = (styles @ styles.T).tocoo()

# Normalization
off_diag_max = similarity_style.data[
    similarity_style.row != similarity_style.col
].max()
similarity_style.data /= off_diag_max

# Re-indexing tp match database
new_shape = (similarity_style.shape[0] + 1, similarity_style.shape[1] + 1)
similarity_style = coo_matrix(
    (
        similarity_style.data,
        (similarity_style.row + 1, similarity_style.col + 1),
    ),
    shape=new_shape,
)


# Sparcity
sparsity = 1.0 - similarity_style.nnz / (
    similarity_style.shape[0] * similarity_style.shape[1]
)
print(f"Sparsity: {sparsity:.2%}")

# Dataframe tranformation
# similarity_style_df = pd.DataFrame(similarity_style.toarray())
# display(similarity_style_df)

Sparsity: 76.38%


### Grade

#### Fuzzy one-hot grade vector

In [16]:
max_grade = boulders_df.grade.max()
grade_df = pd.get_dummies(boulders_df.grade, dtype=np.float32)
grade_df

,0,1,2,3,4,5,6,7,8,9,...,24,25,26,27,28,29,30,31,32,33
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41312,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
41313,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
41314,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
41315,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [17]:
def grade_update(row, max_grade):
    grade_index = row.idxmax()

    if grade_index == 0:
        return row

    values = np.array([0.5, 0.5], dtype=np.float32)
    offsets = np.array([-1, 1])

    for offset, value in zip(offsets, values):
        current_column = grade_index + offset
        if 0 < current_column <= max_grade:
            row[current_column] = value
    return row

grade_df = grade_df.apply(lambda row: grade_update(row, max_grade=max_grade), axis=1)
grade_df.fillna(0, inplace=True)
display(grade_df)


,0,1,2,3,4,5,6,7,8,9,...,24,25,26,27,28,29,30,31,32,33
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.5,1.0,0.5
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.5,1.0,0.5,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.5,1.0,0.5,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.5,1.0,0.5,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.5,1.0,0.5,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41312,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
41313,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
41314,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
41315,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


#### Conversion to sparse matrix

In [18]:
grade = coo_matrix(grade_df)

#### Grade similarity training

In [19]:
from time import perf_counter
from sklearn.metrics.pairwise import cosine_similarity

# Cosine training
start = perf_counter()
similarity_grade = cosine_similarity(grade)
end = perf_counter()
print(f"Cosine calculation time: {end - start:.4f}")


# Re-indexing to match database
start = perf_counter()
coo = coo_matrix(similarity_grade)
end = perf_counter()
print(f"COO conversion time: {end - start:.4f}")

new_shape = (coo.shape[0] + 1, coo.shape[1] + 1)

similarity_grade = coo_matrix(
    (
        coo.data,
        (coo.row + 1, coo.col + 1),
    ),
    shape=new_shape,
)

# Sparcity
sparsity = 1 - similarity_grade.nnz / (
    similarity_grade.shape[0] * similarity_grade.shape[1]
)
print(f"Sparcity: {sparsity:.2f}")

# similarity_grade_df = pd.DataFrame(similarity_grade.toarray())
# display(similarity_grade_df)

Cosine calculation time: 12.2519
COO conversion time: 45.8198
Sparcity: 0.80


## Coherence check

In [20]:
print(type(similarity_ascents))
print(type(similarity_style))
print(type(similarity_grade))

print(similarity_ascents.shape)
print(similarity_style.shape)
print(similarity_grade.shape)

print(similarity_ascents.nnz)
print(similarity_style.nnz)
print(similarity_grade.nnz)

print(similarity_ascents.dtype)
print(similarity_style.dtype)
print(similarity_grade.dtype)

<class 'scipy.sparse._coo.coo_matrix'>
<class 'scipy.sparse._coo.coo_matrix'>
<class 'scipy.sparse._coo.coo_matrix'>
(41318, 41318)
(41318, 41318)
(41318, 41318)
163281651
403301403
349297691
float32
float32
float32


## Data cleaning
Removing all values in similarity_grade < 0.5 (grade difference >= 2)  
Removing of all non zero values in similarity_ascents and similarity_style where similarity_grade == 0

In [21]:
similarity_grade_cleaned = similarity_grade.copy()
similarity_grade_cleaned.data[similarity_grade_cleaned.data < 0.5] = 0
similarity_grade_cleaned.eliminate_zeros()

In [22]:
similarity_grade_cleaned = similarity_grade_cleaned.tocsr()

In [23]:
from scipy.sparse import csr_matrix

def matrix_cleaning(cleaning_matrix: csr_matrix, matrix_to_clean: coo_matrix):
    """Remove all values in matrix_to_clean that are not indexed in cleaning_matrix
    
    :parameters:
    cleaning_matrix: CSR matrix that serves as reference for data existence
    matrix_to_clean: COO matrix from which some indexes are removed"""
    
    # Boolean mask creation (1D array) - Fancy indexing converted to boolean
    # Check if cleaning_matrix contains the indexes of matrix_to_clean
    mask = cleaning_matrix[matrix_to_clean.row, matrix_to_clean.col].A1 != 0

    # Fancy indexing to remove values from matrix_to_clean that are equal to 0 
    # in cleaning_matrix
    new_rows = matrix_to_clean.row[mask]
    new_cols = matrix_to_clean.col[mask]
    new_data = matrix_to_clean.data[mask]

    return csr_matrix(
        (new_data, (new_rows, new_cols)), shape=matrix_to_clean.shape
    )

similarity_ascents_cleaned = matrix_cleaning(similarity_grade_cleaned, similarity_ascents)
similarity_style_cleaned = matrix_cleaning(similarity_grade_cleaned, similarity_style)

In [24]:
print(
    1
    - similarity_ascents.nnz
    / (similarity_ascents.shape[0] * similarity_ascents.shape[1])
)
print(
    1
    - similarity_ascents_cleaned.nnz
    / (
        similarity_ascents_cleaned.shape[0]
        * similarity_ascents_cleaned.shape[1]
    )
)
print(
    1
    - similarity_style.nnz
    / (similarity_style.shape[0] * similarity_style.shape[1])
)
print(
    1
    - similarity_style_cleaned.nnz
    / (similarity_style_cleaned.shape[0] * similarity_style_cleaned.shape[1])
)
print(
    1
    - similarity_grade.nnz
    / (similarity_grade.shape[0] * similarity_grade.shape[1])
)
print(
    1
    - similarity_grade_cleaned.nnz
    / (similarity_grade_cleaned.shape[0] * similarity_grade_cleaned.shape[1])
)

0.9043557644344349
0.977863620318755
0.7637612422693171
0.9601189518985143
0.795394580861312
0.876940571633386


## Matrix saving

In [ ]:
from scipy.sparse import save_npz

save_npz("../../similarity_ascent.npz", similarity_ascents_cleaned)
save_npz("../../similarity_style.npz", similarity_style_cleaned)
save_npz("../../similarity_grade.npz", similarity_grade_cleaned)

## Recommendation example


In [26]:
def recommend_boulders(
    input_boulders, top_n=5, alpha=0.5, beta=0.25, gamma=0.25
):

    ascents = similarity_ascents_cleaned[:, input_boulders].sum(axis=1).A1
    style = similarity_style_cleaned[:, input_boulders].sum(axis=1).A1
    grade = similarity_grade_cleaned[:, input_boulders].sum(axis=1).A1

    ascents[input_boulders] = 0
    style[input_boulders] = 0
    grade[input_boulders] = 0

    sim_scores = alpha * ascents + beta * style + gamma * grade
    
    best_boulders = np.argsort(-sim_scores)[:top_n]
    
    return best_boulders.tolist()


recommendations = recommend_boulders([6735], top_n=10)
print(recommendations)

[4393, 25706, 20198, 4383, 882, 1526, 20735, 22130, 31814, 10033]
